In [1]:
# Basic Machine Learning Models - Notebook 03
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("🤖 Starting Basic Machine Learning Models...")
print("=" * 60)

# Load the data
print("📊 Loading data...")
train_data = pd.read_csv('../data/processed/train_data.csv')
val_data = pd.read_csv('../data/processed/val_data.csv')
test_data = pd.read_csv('../data/processed/test_data.csv')

print(f"Train: {train_data.shape}, Validation: {val_data.shape}, Test: {test_data.shape}")

# Prepare features - Start with simple numeric features
print("\n🔧 Preparing features...")
numeric_features = ['telecommuting', 'has_company_logo', 'has_questions']

X_train = train_data[numeric_features]
y_train = train_data['fraudulent']
X_val = val_data[numeric_features] 
y_val = val_data['fraudulent']
X_test = test_data[numeric_features]
y_test = test_data['fraudulent']

print(f"Features used: {numeric_features}")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Define models to try
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'Logistic Regression': LogisticRegression(random_state=42, class_weight='balanced'),
    'SVM': SVC(random_state=42, class_weight='balanced', probability=True)
}

# Train and evaluate models
print("\n🎯 Training and evaluating models...")
results = {}

for name, model in models.items():
    print(f"\n--- Training {name} ---")
    
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Predict on validation set
    y_pred = model.predict(X_val_scaled)
    y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    
    results[name] = {
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall,
        'f1_score': f1,
        'model': model
    }
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")

# Compare model performance
print("\n" + "=" * 60)
print("📊 MODEL COMPARISON")
print("=" * 60)

best_model = None
best_f1 = 0

for name, metrics in results.items():
    print(f"\n{name}:")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1-Score:  {metrics['f1_score']:.4f}")
    
    if metrics['f1_score'] > best_f1:
        best_f1 = metrics['f1_score']
        best_model = name

print(f"\n🎉 Best Model: {best_model} (F1-Score: {best_f1:.4f})")

# Evaluate best model on test set
print("\n" + "=" * 60)
print("🧪 FINAL TEST EVALUATION")
print("=" * 60)

best_model_obj = results[best_model]['model']
y_test_pred = best_model_obj.predict(X_test_scaled)
y_test_proba = best_model_obj.predict_proba(X_test_scaled)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_test_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

# Feature importance (for Random Forest)
if hasattr(best_model_obj, 'feature_importances_'):
    print("\n🔍 Feature Importance:")
    importance_df = pd.DataFrame({
        'feature': numeric_features,
        'importance': best_model_obj.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(importance_df)
    
    # Plot feature importance
    plt.figure(figsize=(10, 6))
    plt.barh(importance_df['feature'], importance_df['importance'])
    plt.xlabel('Importance')
    plt.title('Feature Importance')
    plt.tight_layout()
    plt.show()

# Analyze predictions
print("\n📈 Prediction Analysis:")
test_with_pred = test_data.copy()
test_with_pred['prediction'] = y_test_pred
test_with_pred['prediction_prob'] = y_test_proba

# Check some examples
print("\n🔍 Sample predictions (first 5):")
sample_preds = test_with_pred[['title', 'fraudulent', 'prediction', 'prediction_prob']].head()
print(sample_preds)

# Save the best model
import joblib
model_path = f'../models/{best_model.lower().replace(" ", "_")}_basic.pkl'
joblib.dump(best_model_obj, model_path)
print(f"\n💾 Best model saved to: {model_path}")

print("\n" + "=" * 60)
print("✅ Basic ML modeling complete!")
print("🚀 Next step: Deep Learning with text features!")

🤖 Starting Basic Machine Learning Models...
📊 Loading data...
Train: (12516, 18), Validation: (2682, 18), Test: (2682, 18)

🔧 Preparing features...
Features used: ['telecommuting', 'has_company_logo', 'has_questions']
X_train shape: (12516, 3), y_train shape: (12516,)

🎯 Training and evaluating models...

--- Training Random Forest ---
Accuracy: 0.8046
Precision: 0.1580
Recall: 0.7000
F1-Score: 0.2578

--- Training Logistic Regression ---
Accuracy: 0.8151
Precision: 0.1697
Recall: 0.7231
F1-Score: 0.2749

--- Training SVM ---
Accuracy: 0.8046
Precision: 0.1580
Recall: 0.7000
F1-Score: 0.2578

📊 MODEL COMPARISON

Random Forest:
  Accuracy:  0.8046
  Precision: 0.1580
  Recall:    0.7000
  F1-Score:  0.2578

Logistic Regression:
  Accuracy:  0.8151
  Precision: 0.1697
  Recall:    0.7231
  F1-Score:  0.2749

SVM:
  Accuracy:  0.8046
  Precision: 0.1580
  Recall:    0.7000
  F1-Score:  0.2578

🎉 Best Model: Logistic Regression (F1-Score: 0.2749)

🧪 FINAL TEST EVALUATION
Classification Rep